# Non Linear Model 
### finale version

Loso on the training set with the complete pipeline and the optimized model



In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier


# =============================
# 1. Charger le train BRUT
# =============================
df = pd.read_csv("data/raw/train.csv")

# Retirer colonnes constantes
cols_to_keep = ["session_id", "trial_number", "TRIAL_TYPE"]
constant_cols = [
    c for c in df.columns
    if c not in cols_to_keep and df[c].nunique() <= 1
]
df = df.drop(columns=constant_cols)

# Colonnes de features
feature_cols = [
    c for c in df.columns
    if c not in ["session_id", "trial_number", "TRIAL_TYPE"]
]

# Normalisation PAR SESSION !!!!! Normalize in the LOSO 
df[feature_cols] = df.groupby("session_id")[feature_cols].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

X = df[feature_cols].values
y_str = df["TRIAL_TYPE"].values
groups = df["session_id"].values

# Encodage labels
le = LabelEncoder()
y = le.fit_transform(y_str)
num_classes = len(le.classes_)


# =============================
# 2. LOSO CV
# =============================
logo = LeaveOneGroupOut()
fold_accs = []

K = 400   # nombre de features à garder

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), start=1):

    X_train_raw = X[train_idx]
    X_test_raw  = X[test_idx]
    y_train = y[train_idx]
    y_test  = y[test_idx]
    test_sessions = np.unique(groups[test_idx])

    # -------------------------------------------
    #  A) Imputation d'abord (pas de NaN pour f_classif)
    # -------------------------------------------
    imputer = KNNImputer(n_neighbors=2)
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    # -------------------------------------------
    #  B) Sélection des 400 meilleures features sur le TRAIN imputé
    # -------------------------------------------
    f_scores, _ = f_classif(X_train_imp, y_train)
    important_idx = np.argsort(f_scores)[::-1][:K]

    X_train_sel = X_train_imp[:, important_idx]
    X_test_sel  = X_test_imp[:, important_idx]

    # -------------------------------------------
    #  C) Modèle XGBoost IDENTIQUE
    # -------------------------------------------
    model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=0,
        n_jobs=-1,
    )

    model.fit(X_train_sel, y_train)
    y_pred = model.predict(X_test_sel)
    acc = accuracy_score(y_test, y_pred)
    fold_accs.append(acc)

    print(f"Fold {fold} — sessions {test_sessions} — accuracy: {acc:.4f}")

# =============================
# 3. Accuracy moyenne LOSO
# =============================
mean_acc = np.mean(fold_accs)
std_acc  = np.std(fold_accs)

print("\n=== RÉSULTAT GLOBAL LOSO ===")
print(f"Accuracy moyenne : {mean_acc:.4f}")
print(f"Écart-type       : {std_acc:.4f}")




Fold 1 — sessions ['PG082_20221113'] — accuracy: 0.5831
Fold 2 — sessions ['PG082_20221114'] — accuracy: 0.6417
Fold 3 — sessions ['PG083_20221125'] — accuracy: 0.5770
Fold 4 — sessions ['PG083_20221129'] — accuracy: 0.5188
Fold 5 — sessions ['PG084_20221206'] — accuracy: 0.5413
Fold 6 — sessions ['PG084_20221207'] — accuracy: 0.5758
Fold 7 — sessions ['PG085_20221213'] — accuracy: 0.4390
Fold 8 — sessions ['PG085_20221214'] — accuracy: 0.5863

=== RÉSULTAT GLOBAL LOSO ===
Accuracy moyenne : 0.5579
Écart-type       : 0.0560


what causes sessions ['PG085_20221213'] to be that bad ?

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA

# =============================
# PARAMÈTRES
# =============================
bad_session = "PG085_20221213"
K = 400

# =============================
# 1) Charger + preprocessing IDENTIQUE au modèle
# =============================
df = pd.read_csv("data/raw/train.csv")

cols_to_keep = ["session_id", "trial_number", "TRIAL_TYPE"]
constant_cols = [
    c for c in df.columns
    if c not in cols_to_keep and df[c].nunique() <= 1
]
df = df.drop(columns=constant_cols)

feature_cols = [
    c for c in df.columns
    if c not in ["session_id", "trial_number", "TRIAL_TYPE"]
]

# normalisation par session
df[feature_cols] = df.groupby("session_id")[feature_cols].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

X = df[feature_cols].values
y_str = df["TRIAL_TYPE"].values
groups = df["session_id"].values

le = LabelEncoder()
y = le.fit_transform(y_str)

# =============================
# 2) Trouver le fold LOSO correspondant
# =============================
logo = LeaveOneGroupOut()

for train_idx, test_idx in logo.split(X, y, groups):
    test_session = np.unique(groups[test_idx])[0]
    if test_session == bad_session:
        break

X_train_raw = X[train_idx]
X_test_raw  = X[test_idx]
y_train = y[train_idx]

# =============================
# 3) Imputation + sélection des 400 features (COMME LE MODÈLE)
# =============================
imputer = KNNImputer(n_neighbors=2)
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp  = imputer.transform(X_test_raw)

f_scores, _ = f_classif(X_train_imp, y_train)
important_idx = np.argsort(f_scores)[::-1][:K]

selected_features = [feature_cols[i] for i in important_idx]

# =============================
# 4) Construire un DF sur ces 400 features
# =============================
df_sel = df[["session_id", "TRIAL_TYPE"] + selected_features]

# =============================
# 5) Diagnostic COMPARATIF
# =============================
rows = []
for s in df_sel["session_id"].unique():
    d = df_sel[df_sel["session_id"] == s]

    Xs = d[selected_features]

    nan_ratio = Xs.isna().mean()
    std_vals  = Xs.std()

    vc = d["TRIAL_TYPE"].value_counts(normalize=True)

    rows.append({
        "session_id": s,
        "n_trials": len(d),
        "nan_mean": nan_ratio.mean(),
        "nan_max": nan_ratio.max(),
        "std_min": std_vals.min(),
        "low_var_features": (std_vals < 1e-6).sum(),
        "top_label_prop": vc.iloc[0],
        "label_entropy": -(vc * np.log(vc + 1e-12)).sum(),
    })

diag_df = pd.DataFrame(rows).set_index("session_id")

print("\n=== BAD SESSION METRICS (ESPACE DU MODÈLE) ===")
print(diag_df.loc[bad_session])

print("\n=== RANG DE LA BAD SESSION ===")
for col, asc in [
    ("nan_mean", False),
    ("low_var_features", False),
    ("std_min", True),
    ("top_label_prop", False),
]:
    rank = diag_df[col].rank(ascending=asc).loc[bad_session]
    print(f"{col:20s} : rank {int(rank)}/{len(diag_df)}")

# =============================
# 6) PCA (optionnel mais très parlant)
# =============================
X_pca = df_sel[selected_features].fillna(
    df_sel[selected_features].mean()
)

coords = PCA(n_components=2, random_state=0).fit_transform(X_pca)

pca_df = pd.DataFrame({
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "session_id": df_sel["session_id"].values
})

mean_pca = pca_df.groupby("session_id")[["PC1", "PC2"]].mean()
bad_dist = ((mean_pca - mean_pca.loc[bad_session])**2).sum(axis=1)

diag_df["pca_dist2_to_bad"] = bad_dist

print("\n=== SESSIONS LES PLUS PROCHES (PCA) ===")
print(diag_df.sort_values("pca_dist2_to_bad").head(5))

print("\n=== SESSIONS LES PLUS ÉLOIGNÉES (PCA) ===")
print(diag_df.sort_values("pca_dist2_to_bad", ascending=False).head(5))



=== BAD SESSION METRICS (ESPACE DU MODÈLE) ===
n_trials            533.000000
nan_mean              0.135000
nan_max               1.000000
std_min               1.000000
low_var_features      0.000000
top_label_prop        0.198874
label_entropy         1.902192
Name: PG085_20221213, dtype: float64

=== RANG DE LA BAD SESSION ===
nan_mean             : rank 4/8
low_var_features     : rank 4/8
std_min              : rank 3/8
top_label_prop       : rank 6/8

=== SESSIONS LES PLUS PROCHES (PCA) ===
                n_trials  nan_mean  nan_max  std_min  low_var_features  \
session_id                                                               
PG085_20221213       533    0.1350      1.0      1.0                 0   
PG084_20221207       752    0.1900      1.0      1.0                 0   
PG084_20221206       508    0.1475      1.0      1.0                 0   
PG082_20221113       662    0.0075      1.0      1.0                 0   
PG083_20221129       453    0.0750      1.0      1.0 

The outcome of this search is that there's no significant difference in this session compare to the others. Search for the best regularization parameters 

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import f_classif
from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

# =============================
# 1. Charger le train BRUT + prétraitement global
# =============================
df = pd.read_csv("data/raw/train.csv")

# Retirer colonnes constantes
cols_to_keep = ["session_id", "trial_number", "TRIAL_TYPE"]
constant_cols = [
    c for c in df.columns
    if c not in cols_to_keep and df[c].nunique() <= 1
]
df = df.drop(columns=constant_cols)

# Colonnes de features
feature_cols = [
    c for c in df.columns
    if c not in ["session_id", "trial_number", "TRIAL_TYPE"]
]

# Normalisation PAR SESSION
df[feature_cols] = df.groupby("session_id")[feature_cols].transform(
    lambda x: (x - x.mean()) / (x.std() + 1e-8)
)

X = df[feature_cols].values
y_str = df["TRIAL_TYPE"].values
groups = df["session_id"].values

# Encodage labels
le = LabelEncoder()
y = le.fit_transform(y_str)
num_classes = len(le.classes_)
print("Classes :", le.classes_)

# =============================
# 2. Pré-calcul des folds LOSO avec KNN + ANOVA(400)
#    -> pour ne pas refaire l'imputation + sélection à chaque set de params
# =============================
logo = LeaveOneGroupOut()
K = 400   # nombre de features à garder

fold_data = []  # liste de dicts : un par fold

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), start=1):
    X_train_raw = X[train_idx]
    X_test_raw  = X[test_idx]
    y_train = y[train_idx]
    y_test  = y[test_idx]
    test_sessions = np.unique(groups[test_idx])

    # --- A) Imputation (pas de NaN pour f_classif)
    imputer = KNNImputer(n_neighbors=2)
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    # --- B) Sélection ANOVA (400 meilleures features sur le TRAIN)
    f_scores, _ = f_classif(X_train_imp, y_train)
    important_idx = np.argsort(f_scores)[::-1][:K]

    X_train_sel = X_train_imp[:, important_idx]
    X_test_sel  = X_test_imp[:, important_idx]

    fold_data.append({
        "X_train": X_train_sel,
        "X_test": X_test_sel,
        "y_train": y_train,
        "y_test": y_test,
        "test_sessions": test_sessions,
    })

    print(f"Préparation fold {fold} — sessions {test_sessions} — OK")

print("\nPréparation des folds terminée.\n")

# =============================
# 3. Grid de paramètres de régularisation à tester
#    (tu peux élargir si c'est assez rapide)
# =============================
param_grid = []

min_child_weight_values = [1, 5]
gamma_values            = [0.0, 0.1]
reg_alpha_values        = [0.0, 0.3, 1.0]   # L1
reg_lambda_values       = [1.0, 5.0]        # L2

for mcw in min_child_weight_values:
    for g in gamma_values:
        for ra in reg_alpha_values:
            for rl in reg_lambda_values:
                param_grid.append({
                    "min_child_weight": mcw,
                    "gamma": g,
                    "reg_alpha": ra,
                    "reg_lambda": rl,
                })

print(f"Nombre de combinaisons à tester : {len(param_grid)}\n")

# =============================
# 4. Boucle sur les combinaisons de régularisation
# =============================
best_mean_acc = -np.inf
best_params = None

for i, params in enumerate(param_grid, start=1):
    fold_accs = []

    print(f"=== Combinaison {i}/{len(param_grid)} : {params} ===")

    for fold_idx, fd in enumerate(fold_data, start=1):
        X_train = fd["X_train"]
        X_test  = fd["X_test"]
        y_train = fd["y_train"]
        y_test  = fd["y_test"]
        test_sessions = fd["test_sessions"]

        # Modèle XGBoost avec tes hyperparams fixes + régularisation variable
        model = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",
            num_class=num_classes,
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=0,
            n_jobs=-1,
            min_child_weight=params["min_child_weight"],
            gamma=params["gamma"],
            reg_alpha=params["reg_alpha"],
            reg_lambda=params["reg_lambda"],
            verbosity=0
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        fold_accs.append(acc)

        print(f"  Fold {fold_idx} — sessions {test_sessions} — acc: {acc:.4f}")

    mean_acc = np.mean(fold_accs)
    print(f"--> Mean LOSO accuracy pour {params} : {mean_acc:.4f}\n")

    if mean_acc > best_mean_acc:
        best_mean_acc = mean_acc
        best_params = params

# =============================
# 5. Résultat final
# =============================
print("====================================")
print("MEILLEURS PARAMÈTRES DE RÉGULARISATION TROUVÉS :")
print(best_params)
print(f"Mean LOSO accuracy correspondant : {best_mean_acc:.4f}")
print("====================================")


Classes : ['GO W+ lick' 'GO W+ nolick' 'GO W- lick' 'GO W- nolick' 'NOGO W+ lick'
 'NOGO W+ nolick' 'NOGO W- lick' 'NOGO W- nolick' 'no tone W+ lick'
 'no tone W+ nolick']
Préparation fold 1 — sessions ['PG082_20221113'] — OK
Préparation fold 2 — sessions ['PG082_20221114'] — OK
Préparation fold 3 — sessions ['PG083_20221125'] — OK
Préparation fold 4 — sessions ['PG083_20221129'] — OK
Préparation fold 5 — sessions ['PG084_20221206'] — OK
Préparation fold 6 — sessions ['PG084_20221207'] — OK
Préparation fold 7 — sessions ['PG085_20221213'] — OK
Préparation fold 8 — sessions ['PG085_20221214'] — OK

Préparation des folds terminée.

Nombre de combinaisons à tester : 24

=== Combinaison 1/24 : {'min_child_weight': 1, 'gamma': 0.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0} ===
  Fold 1 — sessions ['PG082_20221113'] — acc: 0.5831
  Fold 2 — sessions ['PG082_20221114'] — acc: 0.6417
  Fold 3 — sessions ['PG083_20221125'] — acc: 0.5770
  Fold 4 — sessions ['PG083_20221129'] — acc: 0.5188
  Fold 5 —

“We performed an extensive grid search over XGBoost regularization parameters (min_child_weight, gamma, L1/L2 penalties) using LOSO cross-validation.
The optimal configuration yielded only a marginal improvement over the baseline (~+1%), indicating that regularization is not the primary limiting factor for model performance.”

We will now look for better features to add to our pre processed dataset. 